# 🫁 Pneumonia Detection from Chest X-Rays
## EfficientNetB3 + Grad-CAM | Binary Classification

**Architecture:** EfficientNetB3 (Transfer Learning + Fine-Tuning)  
**Task:** Binary — Normal vs Pneumonia  
**Dataset:** Kaggle Chest X-Ray Images (Pneumonia)  
**Explainability:** Grad-CAM heatmaps on detected regions

---
### 📋 Notebook Sections
1. Environment Setup & GPU Check
2. Kaggle Dataset Download
3. Data Exploration & Class Imbalance Analysis
4. Preprocessing Pipeline (CLAHE + Augmentation)
5. Phase 1 — Feature Extraction Training
6. Phase 2 — Fine-Tuning
7. Full Evaluation (Metrics + Confusion Matrix + ROC)
8. Grad-CAM Visualization
9. Live Inference — Upload Your Own X-Ray

## ⚙️ SECTION 1 — Environment Setup & GPU Check

In [ ]:
# ── Verify GPU ──────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU found — go to Runtime > Change runtime type > T4 GPU')

# ── Install / upgrade libraries ─────────────────────────────────────────────
!pip install -q kaggle opencv-python-headless scikit-learn matplotlib seaborn tensorflow

import os, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
from pathlib import Path
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                         ModelCheckpoint)
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# ── Reproducibility seed ─────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'✅ TensorFlow  : {tf.__version__}')
print(f'✅ GPUs visible: {tf.config.list_physical_devices("GPU")}')

## 📦 SECTION 2 — Kaggle Dataset Download

> **One-time setup:** Upload your `kaggle.json` API key when prompted.  
> Get it from → kaggle.com > Account > Create New Token

In [ ]:
from google.colab import files

print('📂 Upload your kaggle.json API key file...')
uploaded = files.upload()   # select kaggle.json from your computer

# Place credentials where Kaggle CLI expects them
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset (~1.2 GB)
print('⬇️  Downloading chest X-ray dataset...')
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/ --unzip

# Confirm directory structure
BASE_DIR = Path('/content/chest_xray')
for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        p = BASE_DIR / split / cls
        count = len(list(p.glob('*.jpeg')) + list(p.glob('*.jpg')) + list(p.glob('*.png')))
        print(f'  {split:5s}/{cls:9s} → {count:5d} images')

print('\n✅ Dataset ready!')

## 🔍 SECTION 3 — Data Exploration & Class Imbalance Analysis

In [ ]:
# ── Count images per class ───────────────────────────────────────────────────
def count_images(split):
    counts = {}
    for cls in ['NORMAL', 'PNEUMONIA']:
        p = BASE_DIR / split / cls
        counts[cls] = len(list(p.glob('*.jpeg')) + list(p.glob('*.jpg')) + list(p.glob('*.png')))
    return counts

train_counts = count_images('train')
val_counts   = count_images('val')
test_counts  = count_images('test')

# ── Class imbalance bar chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Class Distribution per Split', fontsize=14, fontweight='bold')
colors = ['#4CAF50', '#F44336']
for ax, (split, counts) in zip(axes, [('Train', train_counts),
                                        ('Validation', val_counts),
                                        ('Test', test_counts)]):
    bars = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor='black', linewidth=0.7)
    ax.set_title(split, fontweight='bold')
    ax.set_ylabel('Count')
    for bar, val in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(val), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

total_train = sum(train_counts.values())
ratio = train_counts['PNEUMONIA'] / train_counts['NORMAL']
print(f'⚠️  Training imbalance ratio  → {ratio:.2f}:1  (Pneumonia:Normal)')
print(f'📌 Class weights will compensate during training')

# ── Show sample X-rays ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample X-Rays: Normal vs Pneumonia', fontsize=14, fontweight='bold')
for col, cls in enumerate(['NORMAL', 'PNEUMONIA']):
    imgs = list((BASE_DIR / 'train' / cls).glob('*.jpeg'))[:4]
    for row, img_path in enumerate(imgs):
        img = Image.open(img_path).convert('RGB')
        axes[row][col*2].imshow(img, cmap='gray')
        title = f'{cls}\n{img.size[0]}x{img.size[1]}'
        axes[row][col*2].set_title(title, fontsize=9,
                                   color='green' if cls=='NORMAL' else 'red')
        axes[row][col*2].axis('off')

# Remove unused subplot slots
for r in range(2):
    axes[r][1].axis('off')
    axes[r][3].axis('off')
plt.tight_layout()
plt.show()

## 🔧 SECTION 4 — Preprocessing Pipeline

### What we apply:
- **CLAHE** — enhances subtle lung opacities, air bronchograms, and interstitial patterns
- **Resize to 300×300** — EfficientNetB3 native resolution
- **ImageNet normalization** — mean/std matching EfficientNet's pretraining
- **Augmentation (train only)** — flip, rotate, zoom, brightness

In [ ]:
# ── Global config ────────────────────────────────────────────────────────────
IMG_SIZE    = 300        # EfficientNetB3 native size
BATCH_SIZE  = 32
NUM_CLASSES = 1          # Binary sigmoid output
CLASSES     = ['NORMAL', 'PNEUMONIA']

# ── CLAHE preprocessing function ─────────────────────────────────────────────
# CLAHE = Contrast Limited Adaptive Histogram Equalization
# It enhances local contrast so the model can better detect:
#   - consolidations (dense white areas)
#   - interstitial infiltrates (faint net-like patterns)
#   - air bronchograms (subtle dark branching in white areas)

def apply_clahe(img_array):
    """Apply CLAHE to a uint8 numpy array (H, W, 3)."""
    img_uint8 = (img_array * 255).astype(np.uint8) if img_array.max() <= 1.0 else img_array.astype(np.uint8)
    lab = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return enhanced.astype(np.float32) / 255.0

# ── Custom preprocessing for EfficientNetB3 ──────────────────────────────────
def preprocess_fn(img_array):
    """CLAHE → EfficientNet preprocessing."""
    clahe_img = apply_clahe(img_array)
    # EfficientNet expects values in [0, 255] range; it normalises internally
    return tf.keras.applications.efficientnet.preprocess_input(
        clahe_img * 255.0
    )

# ── Visualise CLAHE effect ───────────────────────────────────────────────────
sample_pneumonia = list((BASE_DIR / 'train' / 'PNEUMONIA').glob('*.jpeg'))[5]
orig = np.array(Image.open(sample_pneumonia).resize((IMG_SIZE, IMG_SIZE)).convert('RGB'))
enhanced = apply_clahe(orig)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(orig); ax1.set_title('Original X-Ray', fontweight='bold'); ax1.axis('off')
ax2.imshow(enhanced); ax2.set_title('After CLAHE Enhancement', fontweight='bold'); ax2.axis('off')
fig.suptitle('CLAHE Effect on Chest X-Ray — Enhances Opacity Details', fontsize=12)
plt.tight_layout()
plt.show()
print('💡 CLAHE boosts local contrast → makes subtle infiltrates and consolidations more visible to the CNN')

In [ ]:
# ── Data generators with augmentation ───────────────────────────────────────
# Training augmentation parameters chosen specifically for chest X-rays:
#   - Horizontal flip: YES (left/right lung symmetry is meaningful)
#   - Vertical flip: NO  (lung anatomy has fixed orientation)
#   - Rotation: ±10°    (patient positioning variation)
#   - Zoom: ±10%        (different image scales)
#   - Brightness: ±20%  (exposure variation between machines)
#   - Shear: 5%         (slight tilt variation)

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_fn,
    horizontal_flip=True,
    rotation_range=10,
    zoom_range=0.10,
    brightness_range=[0.8, 1.2],
    shear_range=5,
    fill_mode='reflect'
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_fn
)  # No augmentation on val/test — only CLAHE + normalization

# ── Fix Kaggle dataset's tiny val split by merging with train ────────────────
# The Kaggle dataset only has 16 val images — too small.
# We'll use the full train folder with a 90/10 internal split.
train_gen = train_datagen.flow_from_directory(
    BASE_DIR / 'train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=CLASSES,
    subset='training',
    validation_split=0.10,
    seed=SEED,
    shuffle=True
)

# We need a separate generator with validation_split for val
train_gen_for_val = ImageDataGenerator(
    preprocessing_function=preprocess_fn,
    validation_split=0.10
)

val_gen = train_gen_for_val.flow_from_directory(
    BASE_DIR / 'train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=CLASSES,
    subset='validation',
    seed=SEED,
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    BASE_DIR / 'test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=CLASSES,
    shuffle=False
)

print(f'\n📊 Train batches  : {len(train_gen)} ({train_gen.samples} images)')
print(f'📊 Val batches    : {len(val_gen)} ({val_gen.samples} images)')
print(f'📊 Test batches   : {len(test_gen)} ({test_gen.samples} images)')
print(f'\n🏷️  Class index map: {train_gen.class_indices}')
# NORMAL=0, PNEUMONIA=1

# ── Compute class weights to handle imbalance ────────────────────────────────
labels = train_gen.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
CLASS_WEIGHTS = dict(enumerate(class_weights_array))
print(f'\n⚖️  Class weights  : {CLASS_WEIGHTS}')
print('   → Higher weight on NORMAL class compensates for fewer samples')

## 🧠 SECTION 5 — Model Architecture: EfficientNetB3

```
Input (300×300×3)
    │
    ▼
[EfficientNetB3 backbone — ImageNet pretrained]
    │  1,536 feature channels from top conv layer
    ▼
GlobalAveragePooling2D         → [1,536]
    │
BatchNormalization
    │
Dense(512, ReLU) + Dropout(0.4)
    │
Dense(256, ReLU) + Dropout(0.3)
    │
Dense(1, Sigmoid)              → P(Pneumonia)
```

In [ ]:
def build_model(trainable_base=False, unfreeze_from_layer=None):
    """
    Build EfficientNetB3 classifier.

    Args:
        trainable_base     : If True, unfreeze base model layers
        unfreeze_from_layer: Layer index from which to start unfreezing
                             (None = freeze all / unfreeze all)
    """
    # ── Base model (pretrained on ImageNet) ──────────────────────────────────
    base = EfficientNetB3(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = trainable_base

    # Partial unfreeze for fine-tuning phase
    if trainable_base and unfreeze_from_layer is not None:
        for layer in base.layers[:unfreeze_from_layer]:
            layer.trainable = False
        for layer in base.layers[unfreeze_from_layer:]:
            layer.trainable = True
        frozen = sum(1 for l in base.layers if not l.trainable)
        unfrozen = sum(1 for l in base.layers if l.trainable)
        print(f'  Frozen layers: {frozen} | Unfrozen layers: {unfrozen}')

    # ── Classifier head ──────────────────────────────────────────────────────
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)          # training=False keeps BN frozen
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu', name='dense_512')(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(256, activation='relu', name='dense_256')(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation='sigmoid', name='predictions')(x)

    model = Model(inputs, outputs, name='PneumoniaNet_EfficientB3')
    return model, base

# Build Phase-1 model (frozen base)
model, base_model = build_model(trainable_base=False)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Recall(name='recall'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.AUC(name='auc')
    ]
)

# Summary
total       = sum(np.prod(v.shape) for v in model.trainable_weights)
base_total  = sum(np.prod(v.shape) for v in base_model.weights)
head_total  = sum(np.prod(v.shape) for v in model.trainable_weights)
print(f'\n🏗️  Model built successfully')
print(f'   Base (EfficientNetB3) params : {base_total:,}  [FROZEN in Phase 1]')
print(f'   Classifier head params       : {head_total:,}  [trainable]')
print(f'\n   EfficientNetB3 input size    : {IMG_SIZE}×{IMG_SIZE}')

## 🏋️ SECTION 6 — Phase 1: Feature Extraction Training

Base model is fully **frozen**. Only the classifier head trains.  
This lets the model learn medical-domain patterns without destroying ImageNet features.

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
os.makedirs('/content/checkpoints', exist_ok=True)

callbacks_phase1 = [
    EarlyStopping(
        monitor='val_recall',   # prioritise recall for medical use
        patience=5,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        '/content/checkpoints/phase1_best.keras',
        monitor='val_recall',
        mode='max',
        save_best_only=True,
        verbose=1
    )
]

print('🚀 Phase 1 — Feature Extraction (frozen base)')
print('   Training classifier head only...')
print('   EarlyStopping monitors val_recall (medical priority)\n')

history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_phase1,
    verbose=1
)

print('\n✅ Phase 1 complete!')

In [ ]:
# ── Plot Phase 1 training curves ─────────────────────────────────────────────
def plot_history(history, title='Training History'):
    metrics = ['loss', 'accuracy', 'recall', 'auc']
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for ax, metric in zip(axes.flatten(), metrics):
        ax.plot(history.history[metric], label=f'Train {metric}', linewidth=2)
        ax.plot(history.history[f'val_{metric}'], label=f'Val {metric}',
                linewidth=2, linestyle='--')
        ax.set_title(metric.upper(), fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_history(history1, 'Phase 1 — Feature Extraction')

## 🔓 SECTION 7 — Phase 2: Fine-Tuning

Unfreeze the **last ~40 layers** of EfficientNetB3 and continue with a very small learning rate.  
This adapts the high-level CNN filters to X-ray texture patterns like opacities and consolidations.

In [ ]:
# ── How many layers does the base have? ──────────────────────────────────────
total_layers = len(base_model.layers)
unfreeze_from = total_layers - 40   # unfreeze last 40 layers
print(f'EfficientNetB3 total layers : {total_layers}')
print(f'Unfreezing from layer index : {unfreeze_from}  (last 40 layers)')

# ── Rebuild model with partial unfreeze ──────────────────────────────────────
model, base_model = build_model(
    trainable_base=True,
    unfreeze_from_layer=unfreeze_from
)

# CRITICAL: Use very small LR for fine-tuning to avoid destroying pretrained features
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Recall(name='recall'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.AUC(name='auc')
    ]
)

# Load Phase 1 best weights into the head before fine-tuning
# (Load only classifier weights from phase 1)
phase1_model = keras.models.load_model('/content/checkpoints/phase1_best.keras')
for layer in model.layers:
    if layer.name in ['dense_512', 'dense_256', 'predictions']:
        try:
            layer.set_weights(phase1_model.get_layer(layer.name).get_weights())
        except Exception:
            pass
print('✅ Phase 1 classifier weights transferred to Phase 2 model')

callbacks_phase2 = [
    EarlyStopping(
        monitor='val_recall',
        patience=7,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-8,
        verbose=1
    ),
    ModelCheckpoint(
        '/content/checkpoints/phase2_best.keras',
        monitor='val_recall',
        mode='max',
        save_best_only=True,
        verbose=1
    )
]

print('\n🚀 Phase 2 — Fine-Tuning (last 40 EfficientNet layers unfrozen)')
print('   LR = 1e-5 (very small to preserve pretrained features)\n')

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_phase2,
    verbose=1
)

print('\n✅ Phase 2 complete! Best model saved.')

# Load the best model for evaluation
model = keras.models.load_model('/content/checkpoints/phase2_best.keras')

In [ ]:
plot_history(history2, 'Phase 2 — Fine-Tuning')

## 📊 SECTION 8 — Full Evaluation on Test Set

In [ ]:
print('🧪 Running inference on test set...')
test_gen.reset()

# Get predictions
y_probs = model.predict(test_gen, verbose=1).ravel()
y_true  = test_gen.classes
y_pred  = (y_probs >= 0.5).astype(int)

# ── Classification report ────────────────────────────────────────────────────
print('\n' + '='*55)
print('           CLASSIFICATION REPORT (Test Set)')
print('='*55)
print(classification_report(y_true, y_pred, target_names=CLASSES))

# ── Key metrics summary ──────────────────────────────────────────────────────
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score
acc  = accuracy_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)       # sensitivity (most critical!)
prec = precision_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)
auc  = roc_auc_score(y_true, y_probs)

print(f'\n🎯 FINAL METRICS')
print(f'   Accuracy  : {acc:.4f} ({acc*100:.2f}%)')
print(f'   Recall    : {rec:.4f} ({rec*100:.2f}%)  ← Most important for medical use')
print(f'   Precision : {prec:.4f} ({prec*100:.2f}%)')
print(f'   F1-Score  : {f1:.4f}')
print(f'   AUC-ROC   : {auc:.4f}')

# ── Confusion matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=2, linecolor='white', ax=axes[0])
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=13)
axes[0].set_ylabel('True Label', fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontweight='bold')

# Annotate critical cells
axes[0].text(1.5, 1.5, f'TP={tp}\n(Correct Pneumonia)', ha='center',
             va='center', fontsize=9, color='white', fontweight='bold')
axes[0].text(0.5, 1.5, f'FN={fn}\n(Missed Pneumonia!)', ha='center',
             va='center', fontsize=9, color='darkred')

# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_probs)
axes[1].plot(fpr, tpr, 'b-', linewidth=2.5, label=f'EfficientNetB3 (AUC={auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[1].set_xlabel('False Positive Rate', fontweight='bold')
axes[1].set_ylabel('True Positive Rate (Recall)', fontweight='bold')
axes[1].set_title('ROC Curve', fontweight='bold', fontsize=13)
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n⚠️  False Negatives (missed pneumonia cases): {fn}')
print(f'   These are the most dangerous errors in clinical use.')
print(f'   Consider lowering threshold below 0.5 if FN count is high.')

In [ ]:
# ── Threshold tuning — find optimal threshold for max Recall ────────────────
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)

# Find threshold giving recall >= 0.95
target_recall = 0.95
valid_idx = np.where(recalls[:-1] >= target_recall)[0]
if len(valid_idx) > 0:
    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
    best_thresh = thresholds[best_idx]
    best_prec   = precisions[best_idx]
    best_rec    = recalls[best_idx]
    print(f'🎯 Optimal threshold for Recall ≥ {target_recall}')
    print(f'   Threshold : {best_thresh:.3f}')
    print(f'   Recall    : {best_rec:.4f}')
    print(f'   Precision : {best_prec:.4f}')
    OPTIMAL_THRESHOLD = best_thresh
else:
    OPTIMAL_THRESHOLD = 0.5
    print(f'ℹ️  Using default threshold: 0.5')

# Precision-Recall curve
plt.figure(figsize=(8, 5))
plt.plot(recalls[:-1], precisions[:-1], 'b-', linewidth=2)
plt.axvline(x=target_recall, color='red', linestyle='--', label=f'Target Recall={target_recall}')
if len(valid_idx) > 0:
    plt.scatter([best_rec], [best_prec], color='red', s=100, zorder=5,
                label=f'Optimal point (T={best_thresh:.2f})')
plt.xlabel('Recall (Sensitivity)', fontweight='bold')
plt.ylabel('Precision', fontweight='bold')
plt.title('Precision-Recall Curve — Threshold Selection', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 🔥 SECTION 9 — Grad-CAM Heatmap Visualization

Grad-CAM computes gradients of the predicted class with respect to the **last convolutional layer** of EfficientNetB3.  
This produces a heatmap highlighting **which regions of the X-ray drove the prediction** — letting you verify the model focuses on actual lung opacities, consolidations, and infiltrates.

```
Prediction signal
      │  backprop
      ▼
Last conv layer activations  →  weight by gradient importance  →  heatmap
      │
      ▼
Overlay on original X-ray → clinician sees WHERE the AI detected pathology
```

In [ ]:
class GradCAM:
    """
    Grad-CAM implementation for EfficientNetB3.
    Highlights regions that influenced the pneumonia prediction.
    """

    def __init__(self, model, last_conv_layer_name='top_conv'):
        self.model = model
        # Build a sub-model that outputs both the conv activations and final prediction
        self.grad_model = Model(
            inputs=model.inputs,
            outputs=[
                model.get_layer(last_conv_layer_name).output,
                model.output
            ]
        )

    def compute_heatmap(self, img_array, pred_index=0):
        """Compute Grad-CAM heatmap for a single preprocessed image."""
        with tf.GradientTape() as tape:
            conv_outputs, predictions = self.grad_model(img_array, training=False)
            # For binary classification, use the single output neuron
            loss = predictions[:, pred_index]

        # Gradients of the prediction w.r.t. the last conv feature maps
        grads = tape.gradient(loss, conv_outputs)

        # Pool the gradients over spatial dimensions → importance per channel
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

        # Weight conv outputs by gradient importance
        conv_outputs = conv_outputs[0]   # (H, W, C)
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)

        # ReLU + normalise to [0, 1]
        heatmap = tf.nn.relu(heatmap)
        heatmap = heatmap.numpy()
        if heatmap.max() > 0:
            heatmap /= heatmap.max()
        return heatmap

    def overlay_heatmap(self, original_img, heatmap, alpha=0.45, colormap=cv2.COLORMAP_JET):
        """Superimpose Grad-CAM heatmap on original X-ray."""
        # Resize heatmap to image size
        heatmap_resized = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))
        heatmap_colored = cv2.applyColorMap(
            np.uint8(255 * heatmap_resized), colormap
        )
        heatmap_rgb = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

        # Blend
        superimposed = cv2.addWeighted(
            original_img.astype(np.uint8), 1 - alpha,
            heatmap_rgb, alpha, 0
        )
        return superimposed, heatmap_resized


def preprocess_for_gradcam(img_path, size=IMG_SIZE):
    """Load, CLAHE, and preprocess a single image for model inference."""
    img = Image.open(img_path).convert('RGB').resize((size, size))
    img_array = np.array(img)
    original  = img_array.copy()
    processed = preprocess_fn(img_array)
    return np.expand_dims(processed, 0), original


# Initialise Grad-CAM
# EfficientNetB3's last conv layer is 'top_conv'
gradcam = GradCAM(model, last_conv_layer_name='top_conv')
print('✅ Grad-CAM initialised on layer: top_conv')


def visualise_gradcam(img_path, title='', threshold=OPTIMAL_THRESHOLD):
    """Full Grad-CAM pipeline: load → predict → heatmap → visualise."""
    img_input, original = preprocess_for_gradcam(img_path)

    # Predict
    prob = float(model.predict(img_input, verbose=0)[0][0])
    label = 'PNEUMONIA' if prob >= threshold else 'NORMAL'
    confidence = prob if label == 'PNEUMONIA' else 1 - prob

    # Compute Grad-CAM heatmap
    heatmap = gradcam.compute_heatmap(img_input)
    overlay, heatmap_resized = gradcam.overlay_heatmap(original, heatmap)

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(
        f'{title}\nPrediction: {label}  |  P(Pneumonia)={prob:.4f}  |  Confidence={confidence:.1%}',
        fontsize=13, fontweight='bold',
        color='red' if label=='PNEUMONIA' else 'green'
    )

    axes[0].imshow(original, cmap='gray')
    axes[0].set_title('Original X-Ray', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(heatmap_resized, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap\n(Red = strongest activation)', fontweight='bold')
    axes[1].axis('off')
    plt.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046)

    axes[2].imshow(overlay)
    axes[2].set_title('Overlay\n(Where AI detected pathology)', fontweight='bold')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
    return prob, label

In [ ]:
# ── Visualise Grad-CAM on test samples ───────────────────────────────────────
print('🔥 Generating Grad-CAM heatmaps on test samples...\n')
print('   RED regions = CNN activated strongly → where it sees pathology')
print('   These should align with: consolidations, infiltrates, opacities\n')

# 3 Pneumonia + 2 Normal samples
pneumonia_samples = list((BASE_DIR / 'test' / 'PNEUMONIA').glob('*.jpeg'))[:3]
normal_samples    = list((BASE_DIR / 'test' / 'NORMAL').glob('*.jpeg'))[:2]

for path in pneumonia_samples:
    visualise_gradcam(path, title=f'TRUE: PNEUMONIA | File: {path.name}')

for path in normal_samples:
    visualise_gradcam(path, title=f'TRUE: NORMAL | File: {path.name}')

## 🏥 SECTION 10 — Live Inference: Upload Your Own Chest X-Ray

Upload any chest X-ray image and get:
- Binary prediction (Normal / Pneumonia)
- Probability score
- Grad-CAM heatmap showing the regions that triggered the detection
- Clinical interpretation guide

In [ ]:
from google.colab import files as colab_files

print('📤 Upload a chest X-ray image (JPEG/PNG)...')
uploaded_files = colab_files.upload()

for filename, content in uploaded_files.items():
    # Save locally
    upload_path = f'/content/{filename}'
    with open(upload_path, 'wb') as f:
        f.write(content)

    print(f'\n🔬 Analysing: {filename}')
    print('─' * 50)

    prob, label = visualise_gradcam(
        upload_path,
        title=f'UPLOADED IMAGE: {filename}',
        threshold=OPTIMAL_THRESHOLD
    )

    # ── Clinical interpretation ──────────────────────────────────────────────
    print('\n📋 CLINICAL INTERPRETATION GUIDE')
    print('─' * 50)
    if label == 'PNEUMONIA':
        if prob >= 0.85:
            severity = 'HIGH confidence'
            recommendation = 'Strong radiological evidence of pneumonia. Recommend immediate clinical evaluation.'
        elif prob >= 0.65:
            severity = 'MODERATE confidence'
            recommendation = 'Probable pneumonia. Recommend clinical correlation and possible confirmatory tests.'
        else:
            severity = 'LOW confidence (borderline)'
            recommendation = 'Possible early/mild pneumonia. Clinical correlation strongly advised.'
        print(f'  Result        : ⚠️  PNEUMONIA DETECTED')
        print(f'  Confidence    : {severity}')
        print(f'  P(Pneumonia)  : {prob:.4f} ({prob*100:.1f}%)')
        print(f'  Recommendation: {recommendation}')
    else:
        print(f'  Result        : ✅ NORMAL (No pneumonia detected)')
        print(f'  P(Normal)     : {(1-prob)*100:.1f}%')
        print(f'  Note          : Negative prediction does not fully exclude early-stage')
        print(f'                  or atypical pneumonia. Clinical judgement is essential.')

    print('\n  ⚕️  DISCLAIMER: This AI tool is for research/assistive purposes only.')
    print('      Final diagnosis must be made by a qualified radiologist/physician.')

    # ── What to look for in the heatmap ─────────────────────────────────────
    print('\n🔥 READING THE GRAD-CAM HEATMAP')
    print('─' * 50)
    print('  Red/Yellow areas → Model detected opacity/consolidation here')
    print('  Verify these match known pneumonia signs:')
    print('    ✓ Lobar consolidation  — dense white area in one lobe')
    print('    ✓ Bilateral infiltrates — patchy spots across both lungs')
    print('    ✓ Air bronchograms     — dark branching within white area')
    print('    ✓ Ground-glass opacity — hazy increased density (lower lobes)')

## 💾 SECTION 11 — Save & Export Model

In [ ]:
# ── Save final model ─────────────────────────────────────────────────────────
model.save('/content/pneumonia_efficientnetb3_final.keras')
print('✅ Model saved: /content/pneumonia_efficientnetb3_final.keras')

# ── Download model to your computer ─────────────────────────────────────────
colab_files.download('/content/pneumonia_efficientnetb3_final.keras')

# ── Save training history plots ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for history, phase, ax in [(history1, 'Phase 1', axes[0]),
                            (history2, 'Phase 2', axes[1])]:
    ax.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    ax.plot(history.history['val_recall'],   label='Val Recall',   linewidth=2)
    ax.plot(history.history['val_auc'],      label='Val AUC',      linewidth=2)
    ax.set_title(f'{phase} Validation Metrics', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n🎉 TRAINING COMPLETE!')
print(f'   Model : EfficientNetB3 (Transfer Learning + Fine-Tuning)')
print(f'   Task  : Binary Classification — Normal vs Pneumonia')
print(f'   Extra : Grad-CAM heatmaps for explainability')
print(f'\n   Saved files:')
print(f'   → /content/pneumonia_efficientnetb3_final.keras')
print(f'   → /content/checkpoints/phase1_best.keras')
print(f'   → /content/checkpoints/phase2_best.keras')
print(f'   → /content/training_history.png')